In [ ]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv(override=True)
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL")
) 
MODEL_NAME = "gpt-5-nano"


In [ ]:
def get_weather(city):
    return {
        "城市": city,
        "天气": "晴",
        "温度": "25°C",
        "湿度": "60%",
        "风速": "5 km/h"
    } 

def send_msg(message):
    """
    发送天气提醒给用户
    """
    
    return {
        "status": "success",
        "message": f"已发送消息: {message}"
    }

In [ ]:
# 将工具函数封装成符合规范的工具描述
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "查询指定城市的天气情况，一次只能查询一个城市",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "要查询的城市名称，例如：北京",
                    }
                },
                "required": ["city"]
            },
        }
    },
    {
        "type": "function",
        "function": {
            "name": "send_msg",
            "description": "发送天气提醒给用户",
            "parameters": {
                "type": "object",
                "properties": {
                    "message": {
                        "type": "string",
                        "description": "要发送的消息内容",
                    }
                },
                "required": ["message"]
            },
        }
    },
]

In [ ]:
available_functions = {
    "get_weather": get_weather,
    "send_msg": send_msg
}

In [ ]:
messages = [
    {"role": "user", "content": "请帮我查询北京和上海的天气情况，并将天气结果通知詹姆斯"}
]
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages,
    tools=tools,
    parallel_tool_calls=True,  
)

In [ ]:
response.choices[0].message.tool_calls

In [ ]:
import json
def append_function_messages(messages, response):
    # 拼接第一次大模型请求的返回结果
    messages.append(response.choices[0].message.model_dump())
    tool_calls = response.choices[0].message.tool_calls
    

    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        tool_args = json.loads(tool_call.function.arguments)
        function_to_call = available_functions[tool_name]  # 获取到对应的工具函数

        function_response = function_to_call(**tool_args)
            
        messages.append({
            "role": "tool",   # 固定写法，表示这是工具调用的结果
            "tool_call_id": tool_call.id,   # 关联到具体的工具调用id
            "content": str(function_response)  # 工具执行的结果内容，必须是字符串格式
        })

    return messages

In [ ]:
messages = append_function_messages(messages, response)
messages

In [ ]:
second_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages,
    tools=tools
)

In [ ]:
second_response.choices[0].finish_reason

In [ ]:
second_response.choices[0].message.tool_calls

In [ ]:
messages = append_function_messages(messages, second_response)
messages

In [ ]:
third_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages,
    tools=tools
)

In [ ]:
third_response.choices[0].finish_reason

In [ ]:
from IPython.display import Markdown
display(Markdown(third_response.choices[0].message.content))